# Example 16 — User-friendly quickstart

This notebook shows the high-level `FitSession` API. The low-level PDF/NLL/Minuit classes are still available, but the common workflow is now much shorter.


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    BackgroundSpec, DecayChannel, DecayModel, FitSession, NonResonant,
    Parameter, RealImag, Resonance, enable_x64, plot_dalitz,
    plot_square_dalitz, weighted_resample
)
from dalitzplotfitter.efficiency import FunctionalEfficiency
enable_x64()


## 1. Declare the physics model


In [ ]:
channel = DecayChannel('B+', ('K+', 'pi+', 'pi-'))
nr_x = Parameter.coefficient('NR.x', -0.30, bounds=(-1.2, 0.3), step=0.02, owner='NR')
model = DecayModel(
    channel,
    [
        Resonance('Kstar892', (0,2), RealImag(1.0,0.0), mass=0.8958, width=0.0474, spin=1),
        Resonance('rho770', (1,2), RealImag(0.65,0.10), mass=0.7753, width=0.1491, spin=1),
        NonResonant(RealImag(nr_x,0.10)),
    ],
    normalization_method='square-dalitz', normalization_resolution=250, normalization_pair=(0,2),
)


## 2. Make a toy sample and inspect it with one-line plot helpers


In [ ]:
pool = model.generate_phase_space(100000, seed=16001)
truth = {'NR.x': -0.50}
toy = weighted_resample(jax.random.key(16002), pool, pool.weights*model.intensity(pool.as_dict(), truth), 10000, replace=True)
plot_dalitz(toy, x='s13', y='s23', title='Toy data')
plt.show()
plot_square_dalitz(toy, mother_mass=channel.parent_mass, masses=channel.daughter_masses, pair=(0,2), title='Toy data in Square Dalitz')
plt.show()


## 3. Add efficiency and a background shape

`BackgroundSpec` is automatically normalized on the model quadrature grid. No manual background integral is needed.


In [ ]:
efficiency = FunctionalEfficiency(lambda d: 0.70 + 0.20*jnp.exp(-0.15*d['s13']))
background_shape = lambda d: 1.0 + 0.20*d['s23']
f_sig = Parameter('signal_fraction', 0.85, bounds=(0.0,1.0), step=0.01)
session = FitSession(
    model, toy,
    efficiency=efficiency,
    signal_fraction=f_sig,
    backgrounds=(BackgroundSpec('combinatorial', background_shape),),
)
print([p.name for p in session.parameters])


## 4. Fit in one call


In [ ]:
result = session.fit(start_values={'NR.x': -0.30, 'signal_fraction': 0.85}, simplex=True, ncall=10000, verbose=1)
fit_values = session.print_result(result)
session.print_fit_fractions(result, include_interference=True)


## 5. Plot the fitted signal model


In [ ]:
projection = model.generate_phase_space(120000, seed=16003)
weights = np.asarray(projection.weights * efficiency(projection.as_dict()) * model.intensity(projection.as_dict(), fit_values))
bins = np.linspace(float(toy.s13.min()), float(toy.s13.max()), 70)
plt.figure(figsize=(7,5))
plt.hist(np.asarray(toy.s13), bins=bins, histtype='step', density=True, label='data')
plt.hist(np.asarray(projection.s13), bins=bins, weights=weights, histtype='step', density=True, label='fitted signal model')
plt.xlabel(r'$s_{13}$ [GeV$^2$]'); plt.ylabel('normalized entries'); plt.legend(); plt.show()


## Equivalent ROOT workflow

For real data, the loading and fit can be collapsed to:

```python
session = FitSession.from_root(model, 'data.root', 'DecayTree', s12='S12', s13='S13', s23='S23')
result = session.fit()
```
